# LACE Replication Walkthrough

This notebook provides a lightweight walkthrough for running the main experiment script and inspecting the saved prediction output.

For full reproduction, use `main.py` directly from the command line.

## Running experiments

The main entry point is `main.py`.

Example commands:

```bash
python main.py --dataset Birds --detector LACE --overwrite
python main.py --dataset Birds --detector paper --overwrite
python main.py --dataset all --detector LACE --overwrite
python main.py --dataset all --detector paper --overwrite

### Code: quick run on one small dataset

In [47]:
!python3.8 main.py --dataset Birds --detector LACE --overwrite

Selected datasets: ['Birds']
Selected detectors: ['LACE']
Loading dataset Birds...
Dataset       : Birds
Dataset type  : real
Samples       : 645
Features      : 260
Labels        : 19
Detector      : LACE
Config        : {'n_permutations': 1000, 'seed': 42, 'clustering_method': 'c3m', 'similarity_metric': 'ari', 'matrix_type': 'transposed_window', 'feed_mode': 'unsupervised', 'debug_name': 'LACE', 'window_size': 250, 'step': 125}
Output        : outputs/predictions/Birds/LACE.pkl
Birds | LACE:  70%|██████████████████▏       | 451/645 [00:01<00:00, 380.43it/s]Drift@ 458 (LACE)
Birds | LACE: 100%|██████████████████████████| 645/645 [00:01<00:00, 407.84it/s]
[SAVED] outputs/predictions/Birds/LACE.pkl
Metrics:
  accuracy: 0.2148569155988254
  f1_micro: 0.2545196738745126
  f1_macro: 0.20289139741939585


### Result Inspection

* Choose the dataset-detector pair you would like to inspect. (including _LACE-Agglomerative_Variant_ and _LACE-KMeans_Variant_)

In [ ]:
DATASET = "Birds"
DETECTOR = "LACE"

In [ ]:
from pathlib import Path
import pickle
import numpy as np

result_path = Path(f"outputs/predictions/{DATASET}/{DETECTOR}.pkl")

with open(result_path, "rb") as f:
    result = pickle.load(f)

In [59]:
for metric, value in result["metrics"].items():
    print(f"{metric}: {value:.6f}")

accuracy: 0.214857
f1_micro: 0.254520
f1_macro: 0.202891


In [57]:
print(f"Number of detected drifts: {len(result['drift_points'])}")

for drift in result["drift_points"]:
    print(f"Drift @{drift['stream_idx']}")

Number of detected drifts: 1
Drift @458


### Drift Detection Performance

* MDR and DFD results are given below:

In [65]:
TRUE_DRIFT_POINTS = {
    "Synthetic_Sudden1": [4000, 10000],
    "Synthetic_Sudden2": [4000, 10000],
    "Synthetic_Sudden3": [4000, 10000],
    "Synthetic_Incremental1": [4000, 10000],
    "Synthetic_Incremental2": [4000, 10000],
    "Synthetic_Incremental3": [4000, 10000],
    "Synthetic_Reoccurring": [4000, 10000],
}

In [68]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

prediction_root = Path("outputs/predictions")

drift_records = []

for prediction_file in sorted(prediction_root.glob("*/*.pkl")):
    with open(prediction_file, "rb") as f:
        result = pickle.load(f)
    dataset = result.get("dataset", prediction_file.parent.name)
    detector = result.get("detector", prediction_file.stem)
    if dataset not in TRUE_DRIFT_POINTS:
        continue
    drift_points = result.get("drift_points", [])
    detected_drifts = []
    for drift in drift_points:
        if isinstance(drift, dict):
            detected = drift.get("stream_idx")
        else:
            detected = drift
        if detected is not None:
            detected_drifts.append(int(detected))
    drift_records.append(
        {
            "dataset": dataset,
            "detector": detector,
            "detected_drifts": sorted(detected_drifts),
        }
    )

synthetic_drift_df = pd.DataFrame(drift_records)

def calculate_mdr_dfd(detected_drifts, true_drifts):
    """
    Calculate missed detection rate (MDR) and delay for detection (DFD).

    For each real drift point, the first detected drift after that point is
    counted as the true detection. Detections before the first real drift or
    extra detections after a real drift has already been matched are ignored
    for MDR/DFD, although they are not counted as true detections.

    For every true drift except the last one, only detections before the next
    true drift are considered. This prevents a late detection after the second
    drift from being incorrectly matched to the first drift.
    """

    detected_drifts = sorted(int(x) for x in detected_drifts)
    true_drifts = sorted(int(x) for x in true_drifts)

    matched_detections = []
    delays = []
    missed = 0

    used_detection_indices = set()

    for i, true_drift in enumerate(true_drifts):
        next_true_drift = true_drifts[i + 1] if i + 1 < len(true_drifts) else None

        candidate = None
        candidate_idx = None

        for det_idx, detected_drift in enumerate(detected_drifts):
            if det_idx in used_detection_indices:
                continue

            if detected_drift < true_drift:
                continue

            if next_true_drift is not None and detected_drift >= next_true_drift:
                continue

            candidate = detected_drift
            candidate_idx = det_idx
            break

        if candidate is None:
            missed += 1
            matched_detections.append(None)
            delays.append(np.nan)
        else:
            used_detection_indices.add(candidate_idx)
            matched_detections.append(candidate)
            delays.append(candidate - true_drift)

    n_true_drifts = len(true_drifts)
    mdr = missed / n_true_drifts if n_true_drifts > 0 else np.nan

    valid_delays = [delay for delay in delays if not np.isnan(delay)]
    mean_dfd = float(np.mean(valid_delays)) if valid_delays else np.nan

    return {
        "n_true_drifts": n_true_drifts,
        "n_missed": missed,
        "MDR": mdr,
        "DFD": mean_dfd,
        "matched_detections": matched_detections,
        "per_drift_delay": delays,
    }

mdr_dfd_records = []

for _, row in synthetic_drift_df.iterrows():
    dataset = row["dataset"]
    detector = row["detector"]
    detected_drifts = row["detected_drifts"]
    true_drifts = TRUE_DRIFT_POINTS[dataset]

    scores = calculate_mdr_dfd(
        detected_drifts=detected_drifts,
        true_drifts=true_drifts,
    )

    mdr_dfd_records.append(
        {
            "dataset": dataset,
            "detector": detector,
            "true_drifts": true_drifts,
            "detected_drifts": detected_drifts,
            "matched_detections": scores["matched_detections"],
            "per_drift_delay": scores["per_drift_delay"],
            "n_true_drifts": scores["n_true_drifts"],
            "n_missed": scores["n_missed"],
            "MDR": scores["MDR"],
            "DFD": scores["DFD"],
        }
    )

mdr_dfd_df = pd.DataFrame(mdr_dfd_records)

mdr_table = mdr_dfd_df.pivot_table(
    index="detector",
    columns="dataset",
    values="MDR",
    aggfunc="first",
)

dfd_table = mdr_dfd_df.pivot_table(
    index="detector",
    columns="dataset",
    values="DFD",
    aggfunc="first",
)

mdr_table["Average"] = mdr_table.mean(axis=1, numeric_only=True)
dfd_table["Average"] = dfd_table.mean(axis=1, numeric_only=True)

mdr_table = mdr_table.sort_values("Average", ascending=True)
dfd_table = dfd_table.sort_values("Average", ascending=True)

print("Missed Detection Rate (MDR)")
display(mdr_table.round(4))

print("Delay for Detection (DFD)")
display(dfd_table.round(2))

Missed Detection Rate (MDR)


dataset,Synthetic_Incremental1,Synthetic_Incremental2,Synthetic_Incremental3,Synthetic_Reoccurring,Synthetic_Sudden1,Synthetic_Sudden2,Synthetic_Sudden3,Average
detector,,,,,,,,
LACE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
LD3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
DDM,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.1429
RDDM,0.5,0.5,1.0,0.5,0.5,0.5,0.5,0.5714
LACE-Agglomerative_Variant,1.0,1.0,0.5,0.5,0.5,0.5,0.5,0.6429
ADWIN,1.0,1.0,0.5,0.5,1.0,1.0,1.0,0.8571
HDDM_A,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0000
HDDM_W,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0000
KSWIN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0000


Delay for Detection (DFD)


dataset,Synthetic_Incremental1,Synthetic_Incremental2,Synthetic_Incremental3,Synthetic_Reoccurring,Synthetic_Sudden1,Synthetic_Sudden2,Synthetic_Sudden3,Average
detector,,,,,,,,
RDDM,139.0,29.0,NaN,366.0,114.0,139.0,114.0,150.17
LACE,707.0,119.0,118.5,653.0,669.0,669.0,668.5,514.86
LACE-Agglomerative_Variant,NaN,NaN,30.0,1567.0,563.0,563.0,563.0,657.20
LD3,1118.0,1393.5,1146.0,3621.5,1107.0,673.5,817.5,1411.00
DDM,1776.5,1411.5,NaN,2055.0,1169.0,1172.0,1170.5,1459.08
ADWIN,NaN,NaN,1181.0,3875.0,NaN,NaN,NaN,2528.00
